# Pasokin — Fine-Tuning Model Triase (Opsi B)

Notebook ini fine-tune model kecil open-source (**Gemma 2B**) di atas dataset triase balasan supplier Pasokin, sebagai bukti compliance terhadap rulebook AIC COMPFEST ("Model wajib di fine tune").

**Kenapa bukan Gemini?** Google resmi men-deprecate fine-tuning di Gemini API/AI Studio sejak Mei 2025 — sekarang cuma lewat Vertex AI (berbayar/butuh kartu kredit). Jadi kita pakai model kecil open-source yang bisa di-tune gratis pakai GPU Colab, sebagai bukti proses fine-tuning nyata, sementara production tetap pakai Gemini via in-context learning (ICL).

**Sebelum mulai:**
1. Runtime → Change runtime type → pilih **T4 GPU** (gratis)
2. Upload file dataset kalian (`triage_dataset.jsonl` atau `.csv`) ke sesi Colab lewat panel folder di kiri, ATAU jalankan cell upload di bawah
3. Login Hugging Face (Gemma butuh akses gated model — daftar gratis di huggingface.co, lalu accept license Gemma di halaman modelnya)

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets huggingface_hub

## 2. Login ke Hugging Face
Gemma adalah gated model. Buat token di https://huggingface.co/settings/tokens (read access cukup), lalu accept license di https://huggingface.co/google/gemma-2b-it

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Upload dataset
Format yang diharapkan: JSONL, tiap baris punya field `pesan_balasan` (input) dan `label` / `kategori_triase` (output). Sesuaikan nama kolom di cell parsing kalau beda.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih file dataset kalian
dataset_filename = list(uploaded.keys())[0]
print("File terupload:", dataset_filename)

In [ ]:
import json, pandas as pd

if dataset_filename.endswith(".jsonl") or dataset_filename.endswith(".json"):
    rows = []
    with open(dataset_filename, "r", encoding="utf-8") as f:
        content = f.read().strip()
        try:
            # coba parse sebagai satu array JSON
            rows = json.loads(content)
        except json.JSONDecodeError:
            # fallback: JSONL (satu objek per baris)
            for line in content.splitlines():
                if line.strip():
                    rows.append(json.loads(line))
    df = pd.DataFrame(rows)
else:
    df = pd.read_csv(dataset_filename)

print(f"Jumlah baris: {len(df)}")
print("Kolom:", list(df.columns))
df.head()

## 4. Format dataset jadi prompt instruksi
Dataset Pasokin punya kolom `text_input` (konteks RFQ + balasan supplier mentah) dan `output` (hasil triase dalam bentuk JSON: `classification`, `ai_summary`, `ai_extracted`). Model dilatih untuk generate JSON lengkap ini persis seperti yang dipakai sistem production.

In [ ]:
INPUT_COL = "text_input"
OUTPUT_COL = "output"

SYSTEM_INSTRUCTION = (
    "Kamu adalah asisten triase balasan supplier untuk sistem procurement Pasokin. "
    "Baca konteks RFQ dan balasan supplier, lalu keluarkan HANYA JSON valid dengan field: "
    "classification (confirmed / needs_manual_review), ai_summary (ringkasan singkat), "
    "dan ai_extracted (qty, price, lead_time_days)."
)

PROMPT_TEMPLATE = """<start_of_turn>user
{system}

{input_text}<end_of_turn>
<start_of_turn>model
{output_text}<end_of_turn>"""

def format_row(row):
    return PROMPT_TEMPLATE.format(
        system=SYSTEM_INSTRUCTION,
        input_text=row[INPUT_COL],
        output_text=row[OUTPUT_COL]
    )

df["text"] = df.apply(format_row, axis=1)

from datasets import Dataset
hf_dataset = Dataset.from_pandas(df[["text"]])

split = hf_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} baris | Eval: {len(eval_dataset)} baris")
print("\nContoh formatted:\n")
print(train_dataset[0]["text"])

## 5. Load base model (Gemma 2B, quantized 4-bit biar muat di T4)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

## 6. Setup LoRA (Parameter-Efficient Fine-Tuning)
LoRA dipakai supaya tuning ringan dan cepat di GPU gratis Colab, tapi tetap fine-tuning asli (bobot model berubah, bukan cuma prompt).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./pasokin-triage-gemma",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

trainer.train()

## 8. Simpan model hasil fine-tuning
Ini bukti fisik compliance — LoRA adapter weights hasil training, bisa dilampirkan di proposal/repo.

In [ ]:
trainer.save_model("./pasokin-triage-gemma-final")
tokenizer.save_pretrained("./pasokin-triage-gemma-final")

import shutil
shutil.make_archive("pasokin-triage-gemma-final", "zip", "./pasokin-triage-gemma-final")

from google.colab import files
files.download("pasokin-triage-gemma-final.zip")

## 9. Uji coba cepat (bandingkan sebelum vs sesudah tuning)

In [ ]:
def classify(konteks_dan_balasan):
    prompt = f"""<start_of_turn>user
{SYSTEM_INSTRUCTION}

{konteks_dan_balasan}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    return result.split("model\n")[-1].strip()

test_msg = "Konteks RFQ: Baja Ringan 3000 batang Rp4200. Balasan Supplier: Bisa pak tapi harganya naik jadi 4400 soalnya bahan baku lagi mahal."
print(classify(test_msg))

---
## Catatan untuk proposal
- Notebook ini menghasilkan **LoRA adapter weights** nyata dari base model Gemma 2B — bukti fine-tuning benar-benar dijalankan, bukan simulasi/ICL.
- Screenshot loss curve dari training log (cell 7) bisa dilampirkan sebagai bukti visual di proposal.
- File `pasokin-triage-gemma-final.zip` adalah artifact hasil tuning — simpan sebagai lampiran/bukti di repo.
- Sistem production tetap pakai Gemini via ICL untuk kecepatan & konsistensi output JSON, dengan penjelasan bahwa model fine-tuned ini adalah pembuktian proses tuning sesuai rulebook, dan roadmap ke depan adalah migrasi ke Vertex AI fine-tuning setelah funding/billing tersedia.